In [1]:
import numpy as np
import pandas as pd
import chromadb
import time

# 1. ChromaDB client

In [2]:
df = pd.read_csv("../data/cleaned/cleaned_reviews.csv")
embeddings = np.load("../data/cleaned/sentence_embeddings.npy")

In [3]:
print(df.shape)
print(embeddings.shape)

(30157, 11)
(30157, 384)


In [ ]:
client = chromadb.PersistentClient(path="../data/chromadb")

In [ ]:
client.delete_collection("complaint_embeddings")# For fixing the Three Topic Labelling issues , Don't run thiss

In [6]:
client = chromadb.PersistentClient(path="../data/chromadb")
collection = client.get_or_create_collection(
    name="complaint_embeddings",
    metadata={"hnsw:space":"cosine"} #uses cosine similarity for search
    ) 

In [7]:
print("Collection created: ",collection.name)

Collection created:  complaint_embeddings


In [12]:
# WARNING: Only run this cell once. Collection already persisted to disk.
# Re-running will throw duplicate ID errors.
# To verify: print(collection.count()) — should show 30157

In [8]:
batch_size = 500

for i in range(0,len(df),batch_size):
    batch_df = df.iloc[i:i+batch_size]
    batch_embeddings = embeddings[i:i+batch_size]

    collection.add(
        ids= [str(i) for i in batch_df.index],
        embeddings=batch_embeddings.tolist(),
        documents=batch_df["text_clean_full"].tolist(),
        metadatas= batch_df[['rating','topic_id','topic_label']].to_dict(orient='records')
    )

print("Total documents added: ",collection.count())

Total documents added:  30157


# 2. Retrieval Functions

In [9]:
import re
import emoji
from nltk.corpus import stopwords
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
def text_preprocessing(text,remove_stopwords=False):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+","",text)
    text = re.sub(r'&[a-z]+;|&#\d+;', '', text) 
    text = re.sub(r'<[^>]+>', '', text)
    text = emoji.replace_emoji(text,replace="")
    special_character_pattern = r'[^a-zA-Z0-9\s]+'
    text = re.sub(special_character_pattern,"",text)
    text = re.sub(r"\s+"," ",text).strip()
    if remove_stopwords:
        stop_words = set(stopwords.words("english"))
        words = text.split(" ")
        words = [word for word in words if word not in stop_words]
        text = " ".join(words)
    
    return text

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [11]:
def retrieve(query,collection,model,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances']
    )
    return result

In [12]:
results = retrieve("battery not charging",collection,model,top_k=3)
print(results)

{'ids': [['29641', '6404', '10297']], 'embeddings': None, 'documents': [['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'rating': 1.0, 'topic_label': 'Charging issues', 'topic_id': 0}, {'topic_id': 0, 'rating': 1.0, 'topic_label': 'Charging issues'}, {'topic_label': 'Charging issues', 'topic_id': 0, 'rating': 1.0}]], 'distances': [[0.2036428451538086, 0.2220737338066101, 0.22316217422485352]]}


In [13]:
print("Documents in collection: ",collection.count())

Documents in collection:  30157


In [10]:
def retrieve_filtered(query,collection,model,topic_label,top_k=3):
    query = text_preprocessing(query,remove_stopwords=False)
    query_embedding = model.encode(query).tolist()
    result = collection.query(
        query_embeddings = [query_embedding],
        n_results=top_k,
        include= ['documents','metadatas','distances'],
        where= {"topic_label":{"$eq": topic_label}}
    )
    return result

In [15]:
results = retrieve_filtered("battery not charging",collection,model,topic_label="Charging issues",top_k=3)
print(results)

{'ids': [['29641', '6404', '10297']], 'embeddings': None, 'documents': [['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']], 'uris': None, 'included': ['documents', 'metadatas', 'distances'], 'data': None, 'metadatas': [[{'topic_id': 0, 'topic_label': 'Charging issues', 'rating': 1.0}, {'rating': 1.0, 'topic_id': 0, 'topic_label': 'Charging issues'}, {'rating': 1.0, 'topic_label': 'Charging issues', 'topic_id': 0}]], 'distances': [[0.2036428451538086, 0.2220737338066101, 0.22316217422485352]]}


In [16]:
results_unfiltered = retrieve("battery not charging",collection,model,top_k=3)
results_filtered = retrieve_filtered("battery not charging",collection,model,topic_label="Cracked screen/Display",top_k=3)

print("Unfiltered: ")
print(results_unfiltered['documents'])

print("Filtered: ")
print(results_filtered['documents'])

Unfiltered: 
[['battery will not hold a charge', 'wont charge properly at all', 'does not charge at all']]
Filtered: 
[['the tool to open the phone is too thick so this purchase was a major upset i emailed the seller and they did not respond i will be sending it back since the battery does me no good when i cant open the tablet', 'because of metal parts in the back a phone wont charge on on a pad donated it to my trash can', 'metal part stuck in my phone very cheap and not worth buying unless you need a charger that will only last 2 weeks']]


In [17]:
start = time.time()
results_unfiltered = retrieve("battery not charging",collection,model,top_k=3)
end = time.time()
print(f"Time taken by unfiltered search: {(end-start)*1000} milliseconds")

Time taken by unfiltered search: 32.34720230102539 milliseconds


In [18]:
start = time.time()
results_filtered = retrieve_filtered("battery not charging",collection,model,topic_label="Charging issues",top_k=3)
end = time.time()
print(f"Time taken by filtered search: {(end-start)*1000} milliseconds")

Time taken by filtered search: 110.5351448059082 milliseconds


In [ ]:
# Verifying the Second Label fix
results = retrieve_filtered("camera lens protector blurry",collection,model,topic_label="Camera lens/protector issues",top_k=5)
print(results['documents'])
print(collection.count())

[['the camera screen protector makes pictures blurry with the flash on', 'the camera lenses blur on the sides of the pictures and they dont even have a professional or magnifying effect it works at least if i take my phone case off', 'i could t figure these out it just made my lens blurry i have iphone 8', 'when you put on the camera scratch protector and use the flash the light bleeds through the glass making you photo be nothing but a white blur', 'the protector is too small so its top edge lays close to the front camera distorting the image']]
30157


In [ ]:
# Verifying the Third Label fix
results = retrieve_filtered("scratched easily peeling off",collection,model,topic_label="Generic wear/durability complaints",top_k=5)
print(results['documents'])
print(collection.count())

[['scratches way too easily', 'started peeling off after 2 months', 'scratches easily has already discolored', 'scratches off every easily', 'started peeling after only about 1 month']]
30157


## Phase 5 — ChromaDB Vector Store

### Setup
- **Client:** PersistentClient saved to `data/chromadb/` — persists across sessions
- **Collection:** `complaint_embeddings` with cosine similarity (HNSW index)
- **Documents loaded:** 30,157 complaints in batches of 500
- **Stored per document:** embedding (384-dim), text (`text_clean_full`),
  metadata (`rating`, `topic_id`, `topic_label`)

### Functions Built

**`retrieve(query, collection, model, top_k=3)`**
Semantic search across full corpus. Preprocesses query, embeds it, returns
top-k similar complaints with documents, metadata, and cosine distances.

**`retrieve_filtered(query, collection, model, topic_label, top_k=3)`**
Same as retrieve() but restricted to a specific topic using ChromaDB's
`$eq` metadata filter — ensures retrieved cases match the complaint category.

### Latency Results
| Function | Latency |
|----------|---------|
| `retrieve()` | ~32ms |
| `retrieve_filtered()` | ~110ms |

### Why Filtered Search is Slower
ChromaDB applies metadata filters **post-retrieval** — it searches the full
index first, then filters by metadata. More candidates are evaluated to find
enough matches within the target topic.

**Production fix:** Partition collection by topic — one collection per topic
so vector search is constrained upfront, not filtered after.

### Distance Interpretation
ChromaDB returns cosine **distance** (0=identical, 1=unrelated) — inverse of
cosine similarity. Best match for "battery not charging" scored 0.20 distance
= 80% similarity. This distance metric will inform the Confidence Gate
threshold in Phase 7 (RAG pipeline) — exact threshold to be determined
during that phase's evaluation, not finalized here.

### Key Verification
- Unfiltered "battery not charging" → 3 Charging Issues complaints 
- Filtered to "Cracked screen/Display" → returns screen-topic complaints
  even for a charging query — demonstrates filter override working correctly 

## ChromaDB Re-ingestion After Topic Fix

ChromaDB was originally ingested while `cleaned_reviews.csv` still carried the pre-reduction 79-topic assignments. The old collection was deleted and rebuilt from scratch after the CSV was corrected to 40 topics.

**Why a full rebuild was necessary**: ChromaDB does not support in-place metadata updates on individual documents at scale. The `delete_collection()` + `get_or_create_collection()` pattern (see cell above) is the correct approach — avoids stale metadata and duplicate ID errors from partial re-ingestion.

**Post-fix verification**: `collection.count()` = 30,157, `topic_label` metadata now reflects 40 clean labels. Phase 6 centroid computation confirmed 40 centroids with shape `(384,)`.

## Topic Mislabel Fix — "Stylus/Pen issues" → "Camera lens/protector issues"

**Detected during:** Phase 6 LIME explainability. The heatmap showed a single
word, "protector," dominating the explanation for a review predicted as
"Stylus/Pen issues" — far stronger than any other signal on the entire map,
with no plausible link to styluses.

**Investigation:**
- Searched all 237 reviews under "Stylus/Pen issues" containing "protector"
  (6 matches) — all were camera-lens or camera-screen-protector complaints,
  none mentioned a stylus
- Sampled 15 reviews at random from the full 237-review topic — same result,
  dominated by camera lens/protector complaints

**Root cause:** BERTopic's `reduce_topics(nr_topics=40)` merges topics by
embedding similarity, not semantic correctness. A small camera-protector
cluster was likely merged with (or mislabeled in place of) the original
stylus cluster during reduction, and the inherited topic name didn't reflect
the merged content.

**Fix applied:**
- Relabeled topic_id in `topic_labels` dict and `df['topic_label']`:
  `"Stylus/Pen issues"` → `"Camera lens/protector issues"`
- Re-saved `cleaned_reviews.csv`
- Deleted and rebuilt ChromaDB collection (same pattern as Day 10 fix)
- `topic_centroids` required no change — keyed by integer `topic_id`,
  unaffected by label string changes

**Verification:** `retrieve_filtered("camera lens protector blurry", ...,
topic_label="Camera lens/protector issues")` returns 3 coherent, on-topic
results. `collection.count()` held at 30,157 post-rebuild.

**Takeaway:** LIME wasn't just explaining predictions — it caught a genuine
upstream data-quality bug that survived two prior verification passes
(topic reduction + manual label dictionary review). This is the strongest
practical demonstration of explainability value in the whole project.

## Third Label Fix — ChromaDB Re-ingestion

ChromaDB rebuilt after `topic_labels[17]` was corrected from
`"Camera/Camera lens issues"` to `"Generic wear/durability complaints"`
(see `topic_modelling.ipynb` for full investigation and fix).

**Process:** Same delete + rebuild pattern as the two prior label fixes —
`delete_collection()` → `get_or_create_collection()` → re-ingest in
batches of 500.

**Verification:** `collection.count()` = 30,157 (unchanged).
`retrieve_filtered(..., topic_label="Generic wear/durability complaints")`
returns coherent scratching/peeling/durability complaints, confirming the
relabel propagated correctly into vector store metadata.

**Pattern note:** This is the third label correction surfaced during Phase 6
explainability work (first: 78→40 topic reduction bug; second: "Stylus/Pen
issues" → "Camera lens/protector issues"; third: this one). All three
followed the same fix loop: detect via LIME or manual audit → sample
reviews to confirm → patch `topic_labels` dict in `topic_modelling.ipynb`
→ re-apply to df → re-save CSV → rebuild ChromaDB collection. This
repeated pattern is worth noting in the model card as evidence of a
systematic verification gap in BERTopic's `reduce_topics()` output —
not a one-off fluke.